<div style="text-align: center;">

# **Spring 2026 &mdash; CIS 3813<br>Advanced Data Science<br>(Introduction to Machine Learning)**
### Week 7: Evaluating Classification Models

</div>

**Date:** 09 March 2026
**Time:** 6:00–9:00 PM  
**Instructor:** Dr. Patrick T. Marsh  
**Course Verse:** "He has shown you, O mortal, what is good. And what does the Lord require of you? To act justly and to love mercy and to walk humbly with your God."  &mdash; *Micah 6:8 (NIV)*

---

## **Week 7 Learning Objectives**

By the end of this lecture, you will be able to:


---

## **Today's Outline**
- Lecture
    1. Review of Last Week
    2. A 140 Year-Old Answer
    3. The Confusion Matrix: Seeing the Full Picture
    4. Precision, Recall, and F1-Score
    5. 
- Break (10-15 Minutes)
- Lab (or Homework)
- Review


---

## **Opening Reflection**

> *"The one who states his case first seems right, until the other comes and examines him."*
> **— Proverbs 18:17 (ESV)**

A model that *looks* accurate on the surface can be deeply misleading. A classifier that predicts "no disease" for every patient in a rare-disease dataset might achieve 99% accuracy — and yet be completely useless. This week we learn to **examine our models carefully**, choosing metrics that reveal truth rather than flatter us.

---

## **1.1 Review of Last Week** 

Before evaluating classifiers, let's briefly recall what we built last week.

**Week 6 key ideas:**
- Logistic regression outputs a **probability** (0 to 1) via the **sigmoid function**: $\sigma(z) = \frac{1}{1+e^{-z}}$
- The **log-loss** (binary cross-entropy) is the loss function — not MSE
- A **decision threshold** (default 0.5) converts probabilities into class labels
- Shifting the threshold creates a tradeoff between types of errors

**The big question we left unanswered:** *How do we know if the model is actually good?*

---

## **1.2 A 140-Year-Old Answer to That Question**

In the spring of 1884, Sergeant J.P. Finley of the U.S. Army Signal Corps published the results of the first serious attempt to forecast tornadoes in the United States. He divided the central and eastern U.S. into 18 districts, issued tornado/no-tornado forecasts twice daily, and after 2,803 forecast occasions, reported something remarkable:

> *"Percentage of verification: **96.6%**"*

Ninety-six-point-six percent accuracy. For tornado forecasting in 1884. This was immediately celebrated — until a geologist named G.K. Gilbert read the paper two months later and identified what he called a **"serious fallacy."**

Gilbert's critique was simple: *what would happen if you just predicted "no tornado" every single time, for all 2,803 occasions, without looking at a single weather map?*

The answer: **98.2% accuracy.** Better than Finley's model. No meteorology required.

This is the problem we're solving today. Let's look at Finley's actual numbers.


In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)

In [ ]:
# ── Finley's actual 1884 tornado forecast data ────────────────────────
# Source: Finley (1884) as reported in Murphy (1996), Table 3
# Pooled across all districts and months: 2,803 forecast occasions
#
#                    Observed: Tornado  |  Observed: No Tornado
#  Forecast: Tornado       28          |         72             = 100
#  Forecast: No Tornado    23          |       2680             = 2703
#                          51          |       2752             = 2803

finley_tp, finley_fp = 28,   72
finley_fn, finley_tn = 23, 2680
finley_N = finley_tp + finley_fp + finley_fn + finley_tn

finley_acc_actual  = (finley_tp + finley_tn) / finley_N
# Naive model: always predict "no tornado" → TP=0, FP=0, FN=51, TN=2752
# Its TN = Finley's TN + Finley's FP (those 72 false alarms become correct
# "no tornado" calls when the model never predicts tornado)
naive_tn           = finley_tn + finley_fp   # 2680 + 72 = 2752
finley_acc_naive   = naive_tn / finley_N     # 2752 / 2803

print("═" * 55)
print("  Finley (1884) — Tornado Forecasts, 2803 Occasions")
print("═" * 55)
print(f"  TP (tornado forecast & observed)  : {finley_tp:5d}")
print(f"  FP (forecast tornado, no tornado) : {finley_fp:5d}")
print(f"  FN (missed tornado)               : {finley_fn:5d}")
print(f"  TN (no forecast, no tornado)      : {finley_tn:5d}")
print(f"  N (total)                         : {finley_N:5d}")
print()
print(f"  Finley's reported accuracy        : {finley_acc_actual:.1%}")
print(f"  'Always predict no tornado'       : {finley_acc_naive:.1%}  ← beats Finley!")
print()
print("  Gilbert (1884): 'a serious fallacy'")
print("  The naive model catches ZERO tornadoes.")

# ── Synthetic imbalanced dataset for the rest of the lecture ─────────
# (We need predicted probabilities for ROC/PR curves — Finley's data
#  only gives us a single operating point, not a probability output.)
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    weights=[0.9, 0.1], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

naive_preds = np.zeros(len(y_test), dtype=int)
naive_acc   = accuracy_score(y_test, naive_preds)

lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_acc   = accuracy_score(y_test, lr_preds)

print()
print("─" * 55)
print("  Synthetic dataset (for probability-based metrics below)")
print("─" * 55)
print(f"  Test set class distribution : {np.bincount(y_test)}")
print(f"  Naive 'always predict 0'    : {naive_acc:.3f}")
print(f"  Logistic Regression         : {lr_acc:.3f}")
print("  Same problem, same fallacy — same fix.")


---

## **1.3 The Confusion Matrix: Seeing the Full Picture**

Gilbert's response to Finley introduced what we now call the **confusion matrix** perspective: instead of collapsing everything into one number, *look at all four outcomes separately.*

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | True Positive (TP) ✅ | False Negative (FN) ❌ |
| **Actually Negative** | False Positive (FP) ❌ | True Negative (TN) ✅ |

**Finley's tornado data in this framework:**
- **TP = 28** — Forecast tornado, tornado occurred. *Caught it!*
- **FP = 72** — Forecast tornado, no tornado. *False alarm.*
- **FN = 23** — Forecast no tornado, tornado occurred. *Missed it!* ← the dangerous one
- **TN = 2,680** — Forecast no tornado, no tornado. *Correct quiet day.*

The naive "always say no tornado" model has: **TP = 0, FP = 0, FN = 51, TN = 2,752.**  
It scores *higher* on accuracy — but catches **zero tornadoes.** That TN = 2,752 is doing all the work.

> **Gilbert's insight (1884):** Accuracy conflates two very different things — skill at predicting events *and* skill at predicting non-events. When one class dominates, accuracy is dominated by the majority class and tells you almost nothing about rare-event skill. This is the **"Finley fallacy"** — and it is still one of the most common mistakes in applied machine learning today.

**Translating to a modern domain (fraud detection):**
- **TP** — We flagged a transaction as fraud and it *was* fraud. Correct!
- **TN** — We cleared a legitimate transaction. Correct!
- **FP** — We flagged a legitimate transaction. *False alarm* (customer inconvenience)
- **FN** — We cleared a fraudulent transaction. *Missed it!* (financial loss)

> **Critical insight:** The cost of FP and FN are almost never equal. Your metric choice should reflect which type of error matters more in your domain.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

# ── Panel 1: Finley's actual model (from Table 3) ─────────────────────
ax1 = fig.add_subplot(gs[0])
finley_cm = np.array([[finley_tn, finley_fp],
                       [finley_fn, finley_tp]])
disp1 = ConfusionMatrixDisplay(finley_cm, display_labels=['No Tornado', 'Tornado'])
disp1.plot(ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title(f'Finley (1884) — Tornado Forecasts\nAccuracy = {finley_acc_actual:.1%}', fontsize=10)

# ── Panel 2: Naive "always no tornado" model ──────────────────────────
ax2 = fig.add_subplot(gs[1])
naive_finley_cm = np.array([[finley_tn + finley_fp, 0],
                             [finley_fn + finley_tp, 0]])
disp2 = ConfusionMatrixDisplay(naive_finley_cm, display_labels=['No Tornado', 'Tornado'])
disp2.plot(ax=ax2, colorbar=False, cmap='Reds')
ax2.set_title(f'Naive Model — Always "No Tornado"\nAccuracy = {finley_acc_naive:.1%}  ← higher!', fontsize=10)

# ── Panel 3: Our synthetic LR model (for the rest of the lecture) ─────
ax3 = fig.add_subplot(gs[2])
cm3 = confusion_matrix(y_test, lr_preds)
disp3 = ConfusionMatrixDisplay(cm3, display_labels=['Negative', 'Positive'])
disp3.plot(ax=ax3, colorbar=False, cmap='Greens')
ax3.set_title(f'Synthetic Dataset — Logistic Regression\nAccuracy = {lr_acc:.1%}', fontsize=10)

fig.suptitle(
    'The Finley Fallacy: Higher Accuracy ≠ Better Model\n'
    'The naive model beats Finley on accuracy — but catches zero tornadoes',
    fontsize=12, fontweight='bold', y=1.075
)
plt.show()

print("Key question: If accuracy is misleading, what should we measure instead?")
print("Gilbert, Peirce, and Doolittle each had an answer — and so do we.")
print()

# Extract LR confusion matrix values for use in later cells
tn, fp, fn, tp = confusion_matrix(y_test, lr_preds).ravel()
print(f"Synthetic LR — TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")


---

## **1.4 Beyond Precision & Recall: The Full Metric Family**

The confusion matrix contains four numbers — TP, TN, FP, FN — and from them we can derive a surprisingly large family of metrics. When Gilbert identified the Finley fallacy in 1884, he didn't just critique accuracy — he immediately proposed *better alternatives*. Within six months, three independent researchers (Gilbert, Peirce, and Doolittle) had each invented a different metric to replace it. Many of the metrics we use today — CSI, the Heidke Skill Score, the Peirce Skill Score — are literally their original formulas, rediscovered and renamed decades later.

Let's build the full family, using both Finley's historical data and our synthetic dataset. We'll bring all of these metrics together visually in section 1.8 with the performance diagram.

### **1.4.1 The Complete Confusion Matrix**

For a binary classifier with N total predictions:

| | **Predicted Positive** | **Predicted Negative** | **Row Total** |
|---|---|---|---|
| **Actually Positive** | True Positive (TP) | False Negative (FN) | TP + FN |
| **Actually Negative** | False Positive (FP) | True Negative (TN) | FP + TN |
| **Column Total** | TP + FP | FN + TN | N |

Every metric below is a ratio derived from these four values. The question each metric answers determines when it should be used.



### **1.4.2 Metrics You've Already Seen**

| Metric | Formula | Plain English | Optimize When... |
|--------|---------|---------------|------------------|
| **Accuracy** | $\frac{TP+TN}{N}$ | Overall fraction correct | Classes balanced, costs equal |
| **Precision** | $\frac{TP}{TP+FP}$ | "When I say yes, am I right?" | False positives are costly |
| **Recall (Sensitivity / TPR)** | $\frac{TP}{TP+FN}$ | "Did I catch everything?" | False negatives are costly |
| **F1-Score** | $2 \cdot \frac{P \times R}{P+R}$ | Harmonic mean of P & R | Imbalanced data, no strong preference |

We'll deepen the explanation of these in section 1.5. First, let's meet the rest of the family.


### **1.4.3 The Other Half of the Picture: Specificity and False Rates**

Recall and precision both focus on the **positive class**. But classifiers make decisions about *both* classes, and sometimes you need to know how well the model handles the negative class too.

#### **Specificity (True Negative Rate / TNR)**
*"Of all the actual negatives, how many did we correctly identify?"*
$$\text{Specificity} = \frac{TN}{TN + FP}$$

Specificity is the "recall for the negative class." A COVID test with high specificity correctly clears most healthy patients — meaning few healthy people are told they might have COVID. High specificity → few false alarms on the negative side.

Recall and specificity together give you the full picture of sensitivity vs. selectivity:
- High recall, low specificity: catches almost everything, but raises many false alarms
- Low recall, high specificity: rarely raises false alarms, but misses many true positives

#### **False Positive Rate (FPR)**
*"Of all the actual negatives, how many did we wrongly flag?"*
$$\text{FPR} = \frac{FP}{FP + TN} = 1 - \text{Specificity}$$

FPR is the x-axis of the ROC curve (section 1.6) — knowing its name helps you read that curve more meaningfully. A low FPR means the model has few false alarms among the negative class.

#### **False Negative Rate (FNR) / Miss Rate**
*"Of all actual positives, how many did we miss?"*
$$\text{FNR} = \frac{FN}{FN + TP} = 1 - \text{Recall}$$

FNR is simply the complement of recall. In safety-critical domains (aviation, nuclear, medical devices), FNR is often the primary metric of interest because missed detections are the dangerous failure mode.

#### **False Discovery Rate (FDR)**
*"Of everything I called positive, what fraction was actually negative?"*
$$\text{FDR} = \frac{FP}{FP + TP} = 1 - \text{Precision}$$

FDR is the complement of precision, told from the error perspective. If your spam filter flags 100 emails and 15 are legitimate, FDR = 15%. In genomics and clinical trials, controlling FDR is a central statistical concern — it's the basis of the Benjamini-Hochberg correction for multiple hypothesis testing.

### **1.4.4 Critical Success Index (CSI) — The Threat Score**

Here's a problem that neither accuracy nor F1 handles well. Imagine you're verifying tornado warnings. Tornadoes are extremely rare — on most days, in most districts, nothing happens. TN (correct quiet-day forecasts) will always be enormous. Any metric that includes TN will be dominated by it, making even a useless model look good. This is precisely the Finley problem.

Gilbert's solution in 1884 was radical: **exclude true negatives entirely.**

$$\text{CSI} = \frac{TP}{TP + FP + FN}$$

The denominator is "everything that mattered" — every forecast-tornado occasion (TP + FP) and every missed real tornado (FN). Quiet days where nothing happened and nothing was predicted are simply excluded from the accounting. The model only gets credit for doing something correctly on the days that actually mattered.

**CSI range:** 0 (worst) to 1 (perfect).

**The geometric intuition:** CSI is actually equivalent to the Intersection over Union (IoU) of the predicted positive set and the actual positive set — a metric you may have seen in object detection. The "events that happened" and "events that were predicted" are two sets; CSI measures how much they overlap relative to their union.

**CSI vs. F1:** They look similar but are not the same:
$$F_1 = \frac{2 \cdot TP}{2 \cdot TP + FP + FN} \qquad \text{CSI} = \frac{TP}{TP + FP + FN}$$

F1 gives double weight to TP in the denominator (because it's the harmonic mean of two fractions that both have TP in their numerators). CSI weights everything equally. **CSI is always ≤ F1**, and the gap widens when precision and recall are asymmetric. For symmetric models (precision ≈ recall), they converge. Neither is universally better — they answer subtly different questions about rare-event skill.


### **1.4.5 Frequency Bias (Bias Score)**

$$\text{Bias} = \frac{TP + FP}{TP + FN} = \frac{\text{Number Predicted Positive}}{\text{Number Actually Positive}}$$

Frequency Bias measures whether your model predicts the positive event at the right *rate*, independent of whether the individual predictions are correct.

- **Bias = 1.0**: The model predicts events at exactly the right frequency
- **Bias > 1.0**: Over-prediction — more positive forecasts than actual events (trigger-happy)
- **Bias < 1.0**: Under-prediction — fewer positive forecasts than actual events (too conservative)

> **Critical warning:** Bias = 1.0 does **not** mean the model is good. A model could achieve perfect bias by randomly assigning the right *number* of positive predictions while getting every individual one wrong. Bias measures calibration of frequency, not accuracy of placement. It is a **diagnostic** tool, not a skill score.

**Why it matters:** A weather model that issues tornado warnings at exactly the climatological frequency but mislocates every tornado has Bias = 1.0 and CSI ≈ 0. The bias tells you the systematic tendency; CSI or recall tells you the actual skill. A model with high bias and low CSI is over-warning without skill. A model with low bias and low CSI is under-warning without skill. You need both to diagnose what's wrong.


### **1.4.6 Matthews Correlation Coefficient (MCC)**

The MCC is the metric that the biomedical literature has increasingly been pushing as the single most informative summary statistic for binary classification — particularly on imbalanced data.

$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

**Range:** −1 to +1, where:
- **+1** = perfect predictions
- **0** = no better than random (equivalent to tossing a coin)
- **−1** = perfectly inverted predictions (the model has learned backwards)

**Why MCC over F1?** F1 uses only three of the four cells (TP, FP, FN) — it completely ignores TN. On an imbalanced dataset, this means F1 can look fine even when the model's behavior on the negative class is terrible. MCC uses all four cells symmetrically, making it much harder to game.

Consider: 990 negatives, 10 positives. A model that predicts *everything* as positive gets: F1 = 2(10)/(2·10 + 0 + 990) ≈ 0.02 (bad), but also MCC ≈ 0 (random). Now a model that predicts *everything as negative*: F1 = 0 (no true positives), MCC ≈ 0. MCC correctly identifies both degenerate models as having no skill. Accuracy would call the all-negative model 99% accurate.

**The mathematical insight:** MCC is equivalent to the Pearson correlation coefficient computed between the actual binary labels and the predicted binary labels. A correlation of 0 means no linear relationship — exactly what we want from a metric that correctly identifies random behavior.


### **1.4.6 When to Use What: A Practical Guide**

| Metric | Use It When... | Watch Out For... |
|--------|---------------|------------------|
| **Accuracy** | Classes are balanced | Misleading on imbalanced data |
| **Precision** | False alarms are costly (spam, legal) | Ignores false negatives; can be gamed by under-predicting |
| **Recall / TPR** | Missing events is costly (cancer, fraud, tornadoes) | Can be gamed by predicting all positive |
| **Specificity / TNR** | You care about the negative class performance | Rarely sufficient alone |
| **F1-Score** | Imbalanced data, no strong P/R preference | Ignores TN; can still be misleading on severe imbalance |
| **CSI / Threat Score** | Rare events where TN dominates and should be excluded | Always ≤ F1; originally designed for meteorology |
| **Frequency Bias** | Diagnosing systematic over/under-prediction | Not a skill score — bias=1 ≠ good model |
| **MCC** | Imbalanced data, want the most honest single-number summary | Harder to explain to non-technical audiences |
| \***AUC-ROC** | Comparing models before choosing a threshold | Optimistic on highly imbalanced data (large TN inflates it) |
| \***PR-AUC** | Highly imbalanced data, threshold-free comparison | Less intuitive than ROC; less widely known |


In [ ]:
from sklearn.metrics import matthews_corrcoef

def compute_metrics(tp, tn, fp, fn, label):
    """Compute the full metric family from confusion matrix values."""
    N = tp + tn + fp + fn
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_tpr  = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    accuracy    = (tp + tn) / N
    fpr_val     = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr_val     = fn / (fn + tp) if (fn + tp) > 0 else 0
    fdr         = fp / (fp + tp) if (fp + tp) > 0 else 0
    f1          = 2*(precision*recall_tpr)/(precision+recall_tpr) if (precision+recall_tpr) > 0 else 0
    csi         = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    bias        = (tp + fp) / (tp + fn) if (tp + fn) > 0 else 0
    # MCC formula directly (no sklearn needed — works on raw counts)
    denom = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc   = (tp*tn - fp*fn) / denom if denom > 0 else 0

    print(f"\n{'═'*65}")
    print(f"  {label}")
    print(f"  N={N}  |  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    print(f"{'─'*65}")
    rows = [
        ("Accuracy",                    accuracy,    "(TP+TN)/N"),
        ("Precision",                   precision,   "TP/(TP+FP)"),
        ("Recall / TPR / Sensitivity",  recall_tpr,  "TP/(TP+FN)"),
        ("Specificity / TNR",           specificity, "TN/(TN+FP)"),
        ("False Positive Rate (FPR)",   fpr_val,     "FP/(FP+TN) = 1−Spec"),
        ("False Negative Rate (FNR)",   fnr_val,     "FN/(FN+TP) = 1−Recall"),
        ("False Discovery Rate (FDR)",  fdr,         "FP/(FP+TP) = 1−Precision"),
        ("F1-Score",                    f1,          "2·P·R/(P+R)"),
        ("CSI / Threat Score",          csi,         "TP/(TP+FP+FN)"),
        ("Frequency Bias",              bias,        "(TP+FP)/(TP+FN)"),
        ("MCC",                         mcc,         "[-1,+1] correlation"),
    ]
    for name, val, formula in rows:
        print(f"  {name:<30} {val:>8.4f}  {formula}")
    return dict(tp=tp, tn=tn, fp=fp, fn=fn,
                accuracy=accuracy, precision=precision, recall=recall_tpr,
                specificity=specificity, fpr=fpr_val, fnr=fnr_val, fdr=fdr,
                f1=f1, csi=csi, bias=bias, mcc=mcc)

# ── Finley (1884) — historical baseline ───────────────────────────────
finley_metrics = compute_metrics(
    tp=28, tn=2680, fp=72, fn=23,
    label="Finley (1884) — Tornado Forecasts"
)

# ── Naive 'always no tornado' — what Gilbert compared against ─────────
naive_metrics = compute_metrics(
    tp=0, tn=2752, fp=0, fn=51,
    label="Naive Baseline — Always Predict 'No Tornado'"
)

# ── Synthetic LR — used for probability-based metrics below ───────────
tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_test, lr_preds).ravel()
lr_metrics = compute_metrics(
    tp=tp_s, tn=tn_s, fp=fp_s, fn=fn_s,
    label="Synthetic Dataset — Logistic Regression"
)

# Keep scalar references for later cells
tn, fp, fn, tp = tn_s, fp_s, fn_s, tp_s
accuracy    = lr_metrics['accuracy']
precision   = lr_metrics['precision']
recall_tpr  = lr_metrics['recall']
specificity = lr_metrics['specificity']
fpr_val     = lr_metrics['fpr']
fnr_val     = lr_metrics['fnr']
fdr         = lr_metrics['fdr']
f1          = lr_metrics['f1']
csi         = lr_metrics['csi']
bias        = lr_metrics['bias']
mcc         = lr_metrics['mcc']

print("\n" + "═"*65)
print("  Key observation:")
print(f"  Finley accuracy  = {finley_metrics['accuracy']:.3f}  "
      f"but CSI = {finley_metrics['csi']:.3f}, Recall = {finley_metrics['recall']:.3f}")
print(f"  Naive  accuracy  = {naive_metrics['accuracy']:.3f}  "
      f"but CSI = {naive_metrics['csi']:.3f}, Recall = {naive_metrics['recall']:.3f}")
print("  CSI and Recall correctly identify the naive model as having zero skill.")
print("  Gilbert invented CSI in 1884 for exactly this reason.")


In [ ]:
# Visual: all bounded metrics (0–1) plotted as horizontal bars
# Bias and MCC excluded since they have different scales

bounded_metrics = {
    "Accuracy":           accuracy,
    "Precision":          precision,
    "Recall / TPR":       recall_tpr,
    "Specificity / TNR":  specificity,
    "False Positive Rate": fpr_val,
    "False Neg. Rate":    fnr_val,
    "False Discovery Rate": fdr,
    "F1-Score":           f1,
    "CSI / Threat Score": csi,
}

# Color-code: green = 'higher is better', red = 'lower is better'
colors = [
    'steelblue',   # Accuracy
    'seagreen',    # Precision
    'seagreen',    # Recall
    'seagreen',    # Specificity
    'tomato',      # FPR (lower is better)
    'tomato',      # FNR (lower is better)
    'tomato',      # FDR (lower is better)
    'seagreen',    # F1
    'seagreen',    # CSI
]

names = list(bounded_metrics.keys())
vals  = list(bounded_metrics.values())

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, vals, color=colors, edgecolor='white', height=0.6)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6)
for bar, val in zip(bars, vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Score')
ax.set_title('Full Metric Family — Logistic Regression on Imbalanced Data (90/10 split)\n'
             'Green = higher is better   |   Red = lower is better', fontsize=11)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='seagreen', label='Higher is better'),
                   Patch(facecolor='tomato',   label='Lower is better'),
                   Patch(facecolor='steelblue', label='Context-dependent (Accuracy)')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nFor reference (different scale):")
print(f"  MCC             = {mcc:.4f}  (range -1 to +1; 0 = random)")
print(f"  Frequency Bias  = {bias:.4f}  (1.0 = perfect frequency)")

---

## **1.5 Precision, Recall, and F1-Score**

We introduced these metrics briefly in 1.4, but they're important enough to deserve a slower treatment. Each one asks a fundamentally different question about your model's behavior.


### **1.5.1 Precision — "When I raise the alarm, am I usually right?"**

$$\text{Precision} = \frac{TP}{TP + FP}$$

Think of precision as your model's *credibility* when it makes a positive prediction. A spam filter with 95% precision means that 95 out of every 100 emails it flags as spam really are spam — only 5 are legitimate emails wrongly blocked.

**What low precision costs you:** Every FP is a false alarm. In spam filtering, false alarms annoy users. In legal e-discovery, false alarms waste lawyer hours. In medicine, false alarms lead to unnecessary biopsies and patient anxiety. **The cost of a false alarm is domain-specific — and it's always real.**

**How precision can be gamed:** A model that only makes predictions when it's *extremely* confident can achieve near-perfect precision — by simply refusing to predict positive unless completely certain. Such a model would miss many true positives (low recall). Precision alone doesn't tell you whether the model is actually useful.

> **Finley's tornado model:** Precision = 28/(28+72) = **0.28**. Of every 100 tornado warnings Finley issued, only 28 were real tornadoes. 72 were false alarms. A pretty noisy alarm system.

### **1.5.2 Recall — "Did I find all the needles in the haystack?"**

$$\text{Recall} = \frac{TP}{TP + FN}$$

Also called **Sensitivity**, **True Positive Rate (TPR)**, and **Probability of Detection (POD)**. Recall measures how much of the positive class you actually captured. A cancer screening test with 90% recall catches 90 out of every 100 true cancer cases — and misses 10.

**What low recall costs you:** Every FN is a miss. In cancer screening, a miss is a patient who doesn't get treatment. In fraud detection, a miss is a fraudulent transaction that goes through. In tornado forecasting, a miss is a community that didn't get a warning. **A missed detection can be catastrophic when the event is dangerous.**

**How recall can be gamed:** A model that predicts *everything* as positive has perfect recall (every positive is caught) — but its precision collapses to the base rate. This is exactly what the Finley fallacy is in reverse: optimize for one number without context and you can make the model look arbitrarily good.

> **Finley's tornado model:** Recall = 28/(28+23) = **0.549**. Finley caught about 55% of actual tornadoes. The other 45% went unwarned.


### **1.5.3 The Fundamental Tradeoff**

Precision and recall pull in opposite directions. **Lowering the decision threshold** catches more positives (recall goes up) but also accepts more false alarms (precision goes down). **Raising the threshold** makes the model more selective (precision goes up) but misses more true positives (recall goes down).

This isn't a flaw — it's physics. There is no free lunch: you cannot simultaneously maximize both without a better underlying model. The tradeoff exists because **the positive and negative distributions overlap** in feature space, and any threshold cuts through that overlap region.

The question isn't "which is higher?" — it's "which mistake is more expensive in this context?"

| Context | The Costly Error | Prioritize |
|---------|------------------|------------|
| Cancer screening | Missing a cancer (FN) | **Recall** |
| Spam filter | Blocking legitimate email (FP) | **Precision** |
| Tornado warning | Missing a tornado (FN) | **Recall** |
| Legal document review | Including irrelevant docs (FP) | **Precision** |
| Airport security | Missing a threat (FN) | **Recall** |
| Loan approval | Approving a defaulter (FP) | **Precision** |


### **1.5.4 F1-Score — When You Can't Prioritize Either**

$$F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

F1 is the **harmonic mean** of precision and recall. It gives a single number that balances both — useful when you don't have domain knowledge to prefer one over the other, or when you need a single number for model comparison.

**Why harmonic mean, not arithmetic mean?**  
The arithmetic mean rewards extremes: a model with Precision = 1.0 and Recall = 0.0 would average to 0.5 — which sounds decent. The harmonic mean punishes extremes: the same model gets F1 = 0. This is the right behavior, because a model that catches nothing (or flags everything) is not "50% useful" — it's broken.

**F1's blind spot:** F1 completely ignores TN. This means F1 can look good on imbalanced datasets even when the model handles the negative class poorly. On highly imbalanced data, consider MCC (section 1.3.5) instead, which incorporates all four confusion matrix cells.

> **Generalization:** The $F_\beta$ score lets you weight recall $\beta$ times more heavily than precision: $F_\beta = (1+\beta^2) \cdot \frac{P \cdot R}{\beta^2 \cdot P + R}$. Use $F_2$ when recall matters more (medical), $F_{0.5}$ when precision matters more (legal).


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Manual calculation first — always build intuition before using library functions
precision_manual = tp / (tp + fp)
recall_manual    = tp / (tp + fn)
f1_manual        = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)

print("=== Manually Calculated ===")
print(f"  Precision : {precision_manual:.4f}")
print(f"  Recall    : {recall_manual:.4f}")
print(f"  F1-Score  : {f1_manual:.4f}")

print("\n=== Verified with sklearn ===")
print(f"  Precision : {precision_score(y_test, lr_preds):.4f}")
print(f"  Recall    : {recall_score(y_test, lr_preds):.4f}")
print(f"  F1-Score  : {f1_score(y_test, lr_preds):.4f}")

print("\n=== Full Classification Report ===")
print(classification_report(y_test, lr_preds, target_names=['Negative', 'Positive']))

---

## **1.6 The Precision-Recall Tradeoff**

In section 1.5 we described the tension between precision and recall conceptually. Now let's see it in action by sweeping across all possible thresholds and plotting what happens.


### **1.6.1 Why sweep thresholds?**

Logistic regression doesn't actually output class labels — it outputs **probabilities**. The label you get (0 or 1) depends on where you draw the line. The default is 0.5, but there's nothing sacred about that number. It was chosen for mathematical convenience, not because your business problem has symmetric costs.

Consider a medical test: would you rather your model say "probably cancer" at 50% confidence, or wait until it's 80% sure? The answer depends on the consequences — and *you* should make that call, not sklearn's default.


### **1.6.2 Reading the Precision-Recall Curve**

The PR curve plots precision (y-axis) against recall (x-axis) as you sweep the threshold from very high (predict positive rarely, high precision / low recall) to very low (predict positive almost always, low precision / high recall).

**Key features to look for:**
- **A curve that stays high and to the right** is a good model — it maintains high precision even as recall increases
- **A curve that drops sharply as recall increases** means the model quickly runs into false alarms when pushed to catch more positives
- **The "elbow"** of the curve — where precision starts dropping fast — is often a good operating region
- **A flat horizontal line at the base rate** would be a random model — it achieves the positive class frequency as precision regardless of threshold

### **1.6.3 PR-AUC: Summarizing the Whole Curve**

Just as ROC-AUC summarizes the ROC curve, **PR-AUC** (also called Average Precision, or AP) summarizes the PR curve. It answers: *"On average, across all recall levels, what precision does this model achieve?"*

PR-AUC is particularly valuable on **highly imbalanced datasets** where ROC-AUC can be optimistic. When the negative class is enormous, even a mediocre model can maintain a low FPR (making the ROC curve look good), while the PR curve correctly reflects that the model's precision collapses at even modest recall levels.

> **Rule of thumb:** When your positive class is less than ~10% of the data, prefer PR-AUC over ROC-AUC for model selection.


### **1.6.4 The Threshold is a Business Decision**

The code below plots the full precision-recall curve and marks several threshold values. Notice that the *shape* of the curve is fixed by the model — what changes is which point on the curve you operate at. Choosing an operating threshold is not a modeling problem. It's a values problem: how much do you care about false alarms vs. missed detections?

This is one of the places where data science intersects directly with ethics. A threshold chosen to minimize a loss function might not be the threshold that minimizes harm to real people.


In [ ]:
y_proba = lr.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.05, 0.95, 60)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, preds, zero_division=0))
    recalls.append(recall_score(y_test, preds, zero_division=0))
    f1s.append(f1_score(y_test, preds, zero_division=0))

best_idx = np.argmax(f1s)
best_t   = thresholds[best_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(thresholds, precisions, 'b-',  lw=2, label='Precision')
ax1.plot(thresholds, recalls,    'r-',  lw=2, label='Recall')
ax1.plot(thresholds, f1s,        'g--', lw=2, label='F1-Score')
ax1.axvline(best_t, color='gray', ls=':', lw=1.5, label=f'Best F1 @ t={best_t:.2f}')
ax1.set(xlabel='Threshold', ylabel='Score', title='Metrics vs. Decision Threshold')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(recalls, precisions, 'purple', lw=2)
for mark in [0.2, 0.5, 0.7]:
    i = np.argmin(np.abs(thresholds - mark))
    ax2.scatter(recalls[i], precisions[i], s=70, color='purple', zorder=5)
    ax2.annotate(f't={mark}', xy=(recalls[i], precisions[i]),
                 xytext=(recalls[i]+0.04, precisions[i]-0.07), fontsize=8.5, color='purple',
                 arrowprops=dict(arrowstyle='->', color='purple', lw=1))
ax2.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best F1 = {f1s[best_idx]:.3f} at threshold = {best_t:.2f}")

---

## **1.7 ROC Curves and AUC**

### **1.7.1 Where ROC Curves Come From**

The ROC curve has an unusual origin: it was developed by radar engineers and statisticians during World War II to evaluate how well radar operators could distinguish enemy aircraft from noise. The tradeoff they faced was the same one we face: set the sensitivity too high and you get constant false alarms; set it too low and you miss real threats. The "Receiver Operating Characteristic" name comes directly from that signal-detection context.

The same mathematical framework turns out to describe every binary classifier — because every binary classification problem is, at its core, a signal-detection problem.


### **1.7.2 What the Curve Shows**

The ROC curve plots **True Positive Rate (Recall)** on the y-axis against **False Positive Rate** on the x-axis, across every possible decision threshold:

$$\text{TPR} = \frac{TP}{TP+FN} \qquad \text{FPR} = \frac{FP}{FP+TN}$$

At threshold = 1.0 (predict nothing positive): TPR = 0, FPR = 0. Bottom-left corner.  
At threshold = 0.0 (predict everything positive): TPR = 1, FPR = 1. Top-right corner.  
Every threshold in between traces a path from bottom-left to top-right.

**A perfect classifier** would jump immediately to the top-left corner (TPR = 1, FPR = 0) — catching all positives before generating any false positives.  
**A random classifier** would trace the diagonal line — at any threshold, it catches true positives and false positives at the same rate.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, roc_thresh = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

# Random baseline
rng_proba = np.random.RandomState(0).uniform(0, 1, len(y_test))
fpr_r, tpr_r, _ = roc_curve(y_test, rng_proba)

# Optimal threshold (Youden's J: maximizes TPR - FPR)
opt_idx = np.argmax(tpr - fpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr,   tpr,   'b-',  lw=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot(fpr_r, tpr_r, 'r--', lw=1.5, label='Random Classifier (AUC ≈ 0.50)')
ax.plot([0,1], [0,1], 'k:',  lw=1)
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.scatter(fpr[opt_idx], tpr[opt_idx], color='red', s=100, zorder=5,
           label=f'Optimal point (t ≈ {roc_thresh[opt_idx]:.2f})')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate (Recall)',
       title='ROC Curve')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"AUC: {auc:.4f}")

### **1.7.3 AUC: A Threshold-Independent Summary**

AUC (Area Under the Curve) compresses the entire ROC curve into a single number between 0 and 1.

**The probabilistic interpretation:** AUC equals the probability that a randomly chosen positive example receives a *higher predicted score* than a randomly chosen negative example. An AUC of 0.85 means: if you pick one patient with diabetes and one without, the model gives the diabetes patient a higher probability score 85% of the time.

This is a powerful property: AUC measures the model's **discriminative ability** without committing to any threshold. It answers the question "how good is this model at *ranking* positives above negatives?" — which is often more fundamental than "what's its accuracy at threshold X?"

| AUC | Interpretation |
|-----|----------------|
| 1.0 | Perfect — positives always scored higher than negatives |
| 0.9–1.0 | Excellent |
| 0.8–0.9 | Good |
| 0.7–0.8 | Fair |
| 0.6–0.7 | Poor |
| 0.5 | Random — model has no discriminative ability |
| < 0.5 | Worse than random (model has learned backwards) |


### **1.7.4 When to Use ROC-AUC (and When Not To)**

**Use ROC-AUC when:**
- Comparing multiple models before choosing a deployment threshold
- Your classes are roughly balanced
- You care about the model's ranking ability across the full range of thresholds

**Be cautious with ROC-AUC when:**
- Your dataset is highly imbalanced — ROC-AUC can look artificially good because FPR uses the large TN denominator. A model might achieve FPR = 0.01 simply because TN is enormous, even while precision is terrible.
- You need to communicate to a non-technical audience — the probabilistic AUC interpretation is hard to explain; precision and recall are more intuitive.

**In those cases, use PR-AUC instead** (section 1.5) — it ignores TN entirely and better reflects performance on the rare positive class.

### **1.7.5 Youden's J: Choosing the Optimal Threshold from the ROC Curve**

Once you've decided to use a model, you need an operating threshold. **Youden's J statistic** identifies the point on the ROC curve that maximizes the sum of sensitivity and specificity:

$$J = \text{TPR} - \text{FPR} = \text{Sensitivity} + \text{Specificity} - 1$$

It finds the threshold where you get the biggest gap between the true positive rate and the false positive rate — a reasonable default when you have no domain-specific cost information. The code below marks this point on the ROC curve.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, roc_thresh = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

# Random baseline
rng_proba = np.random.RandomState(0).uniform(0, 1, len(y_test))
fpr_r, tpr_r, _ = roc_curve(y_test, rng_proba)

# Optimal threshold (Youden's J: maximizes TPR - FPR)
opt_idx = np.argmax(tpr - fpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr,   tpr,   'b-',  lw=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot(fpr_r, tpr_r, 'r--', lw=1.5, label='Random Classifier (AUC ≈ 0.50)')
ax.plot([0,1], [0,1], 'k:',  lw=1)
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.scatter(fpr[opt_idx], tpr[opt_idx], color='red', s=100, zorder=5,
           label=f'Optimal point (t ≈ {roc_thresh[opt_idx]:.2f})')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate (Recall)',
       title='ROC Curve')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"AUC: {auc:.4f}")

---

## **1.7 Handling Imbalanced Datasets**

### **1.7.1 Why Imbalance Matters**

Real-world classification problems are almost never balanced. Fraud affects roughly 0.1% of transactions. Serious equipment failures might happen in 2% of machinery-hours. Cancer is present in a small fraction of screening patients. Network intrusion events are rare against a background of normal traffic.

Standard machine learning algorithms are trained to minimize overall error — which means they implicitly optimize for accuracy. On a dataset that's 99% negative, a model that *always predicts negative* has 99% accuracy and zero usefulness. We've already seen this with Finley's data. Now let's fix it.

The core problem is that the **gradient signal** from minority-class examples is drowned out during training. With 990 negatives and 10 positives in a batch, each positive example contributes less than 1% of the gradient update. The model simply doesn't learn much from the class that matters most.


### **1.7.2 Strategy 1: Class Weights (Today's Focus)**

The cleanest solution is to tell the algorithm: *"mistakes on the minority class should cost more."*

With `class_weight='balanced'`, sklearn automatically computes weights inversely proportional to class frequency:

$$w_c = \frac{N}{k \cdot N_c}$$

where $N$ is total samples, $k$ is number of classes, and $N_c$ is the count of class $c$. On a 90/10 split, the positive class gets a weight of ~5× — meaning each missed positive counts as much as five missed negatives in the loss function.

**Advantages:** No change to the data, no synthetic samples, simple one-parameter change. Works well in most cases.  
**Disadvantage:** Doesn't help if the minority class simply doesn't have enough examples to learn from.


### **1.7.3 Strategy 2: Oversampling the Minority Class**

Rather than reweighting, you can **duplicate or synthesize** minority-class examples until the classes are balanced. The most widely used technique is **SMOTE** (Synthetic Minority Over-sampling Technique), which creates new synthetic examples by interpolating between existing minority-class neighbors in feature space.

**Advantages:** Creates genuinely new training examples, not just copies. Can significantly improve performance when the minority class is very small.  
**Disadvantage:** Risk of overfitting to the synthetic examples; adds complexity; can introduce noise if the minority class is not well-clustered.

### **1.7.4 Strategy 3: Undersampling the Majority Class**

Instead of adding minority examples, you **randomly drop** majority-class examples until the ratio is more balanced. Simple and fast, but you're throwing away real data — which is wasteful and can hurt performance if the majority class contains important variability.

In [ ]:
# Compare default vs. balanced class weights
lr_default  = LogisticRegression(random_state=42)
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42)

lr_default.fit(X_train, y_train);  preds_default  = lr_default.predict(X_test)
lr_balanced.fit(X_train, y_train); preds_balanced = lr_balanced.predict(X_test)

print("=== Default Weights ===")
print(classification_report(y_test, preds_default, target_names=['Negative', 'Positive']))

print("=== Balanced Weights ===")
print(classification_report(y_test, preds_balanced, target_names=['Negative', 'Positive']))

print("Note: Balanced weights improve recall on the minority class,")
print("but lower precision — more false alarms. This tradeoff is deliberate.")

### **1.7.5 Choosing a Strategy**

| Situation | Recommended Approach |
|-----------|----------------------|
| Mild imbalance (e.g., 80/20) | Class weights usually sufficient |
| Moderate imbalance (e.g., 90/10) | Class weights; try SMOTE if still underperforming |
| Severe imbalance (e.g., 99/1) | SMOTE + class weights; consider anomaly detection methods |
| Very small minority class (< 100 examples) | Get more data if possible; SMOTE with caution |


### **1.7.6 What Changes — and What Doesn't**

Class weighting changes the model's **decision boundary** — it shifts the boundary toward the majority class, making the model more willing to predict positive. This improves recall at the cost of precision. The underlying predicted probabilities also shift: the balanced model assigns higher probabilities to positive examples on average.

Importantly, **no strategy eliminates the precision-recall tradeoff** — they just move you to a different point on the curve. The curve itself only improves if you add more informative features or more data.


In [ ]:
# Compare default vs. balanced class weights
lr_default  = LogisticRegression(random_state=42)
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42)

lr_default.fit(X_train, y_train);  preds_default  = lr_default.predict(X_test)
lr_balanced.fit(X_train, y_train); preds_balanced = lr_balanced.predict(X_test)

print("=== Default Weights ===")
print(classification_report(y_test, preds_default, target_names=['Negative', 'Positive']))

print("=== Balanced Weights ===")
print(classification_report(y_test, preds_balanced, target_names=['Negative', 'Positive']))

print("Note: Balanced weights improve recall on the minority class,")
print("but lower precision — more false alarms. This tradeoff is deliberate.")

In [ ]:
# Visualize probability distributions by class under each model
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, preds, title in zip(
    axes,
    [lr_default, lr_balanced],
    [preds_default, preds_balanced],
    ['Default Weights', 'Balanced Weights']
):
    proba = model.predict_proba(X_test)[:, 1]
    ax.hist(proba[y_test == 0], bins=25, alpha=0.6, color='steelblue', label='True Negative')
    ax.hist(proba[y_test == 1], bins=25, alpha=0.6, color='tomato',    label='True Positive')
    ax.axvline(0.5, color='black', ls='--', lw=1.5, label='Threshold = 0.5')
    rec = recall_score(y_test, preds)
    pre = precision_score(y_test, preds)
    ax.set_title(f'{title}  |  Recall={rec:.2f}  Precision={pre:.2f}')
    ax.set_xlabel('Predicted Probability (Class 1)')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Class Weighting Shifts the Probability Distribution',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## **1.8 The Performance Diagram**

You've now seen two threshold-free visualizations:
- **The PR curve** (section 1.5): Precision vs. Recall — emphasizes rare-event detection, ignores TN
- **The ROC curve** (section 1.6): TPR vs. FPR — uses all four cells, good for general comparison

The **performance diagram** (Roebber 2009) is a third member of this family, designed specifically for rare-event problems where the ROC curve's use of FPR (which depends on the huge TN count) can be misleading.

#### **How the axes are defined**

| Axis | Metric | Formula |
|------|--------|---------|
| **x-axis** | Success Ratio (SR) | $\\frac{TP}{TP+FP}$ = **Precision** |
| **y-axis** | Probability of Detection (POD) | $\\frac{TP}{TP+FN}$ = **Recall** |

SR is just another name for Precision; POD is another name for Recall/TPR. The renaming is conventional in the meteorological verification literature — the same literature where Gilbert invented CSI in response to Finley's 1884 tornado data.

#### **What's overlaid**

- **CSI isolines** — curved lines connecting all (SR, POD) combinations with the same CSI value. Moving toward the top-right corner means higher CSI.
- **Frequency Bias isolines** — straight lines from the origin. The 45° line is Bias = 1 (perfect frequency). Above it = over-prediction; below = under-prediction.

The **ideal point is the top-right corner** (SR = 1, POD = 1): no false positives, no false negatives, CSI = 1, Bias = 1.

#### **Performance diagram vs. ROC curve vs. PR curve**

| | ROC Curve | PR Curve | Performance Diagram |
|--|-----------|----------|---------------------|
| x-axis | FPR (uses TN) | Recall | Success Ratio (= Precision) |
| y-axis | Recall | Precision | POD (= Recall) |
| Overlaid | — | — | CSI + Bias isolines |
| Best point | Top-left | Top-right | Top-right |
| TN included? | Yes | No | No |
| Best for | General ML | Imbalanced | Rare events, meteorology |

The key difference from ROC: the x-axis uses Precision (ignores TN) instead of FPR (requires TN). When TN is enormous — as in tornado forecasting, fraud detection, or any rare-event problem — ROC's x-axis is dominated by that huge TN denominator and can look artificially good. The performance diagram sidesteps this entirely.

> **Connecting to Proverbs 18:17:** Each of these three diagrams examines the model from a different angle. The ROC curve examines it through the lens of false alarm rate. The PR curve examines it through reliability. The performance diagram examines it through rare-event skill. None alone tells the complete story — together, they reveal different facets of the truth.

In [ ]:
import matplotlib.ticker as ticker
from sklearn.metrics import precision_score as ps, recall_score as rs

# Note: lr_default, lr_balanced, preds_default, preds_balanced
# are all already defined from section 1.7 above.

# ── Helper: CSI from SR (Precision) and POD (Recall) ─────────────────
def csi_from_sr_pod(sr, pod):
    with np.errstate(divide='ignore', invalid='ignore'):
        result = np.where(
            (sr > 0) & (pod > 0),
            1.0 / (1.0/sr + 1.0/pod - 1.0),
            0.0
        )
    return np.clip(result, 0, 1)

# ── Grid for CSI isolines ─────────────────────────────────────────────
sr_grid  = np.linspace(0.001, 1.0, 500)
pod_grid = np.linspace(0.001, 1.0, 500)
SR, POD  = np.meshgrid(sr_grid, pod_grid)
CSI_grid = csi_from_sr_pod(SR, POD)

fig, ax = plt.subplots(figsize=(8, 7))

# CSI filled contours
csi_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
cf = ax.contourf(SR, POD, CSI_grid, levels=[0]+csi_levels+[1.0],
                 cmap='YlGn', alpha=0.45)
cs = ax.contour(SR, POD, CSI_grid, levels=csi_levels,
                colors='darkgreen', linewidths=0.8, alpha=0.7)
ax.clabel(cs, fmt='CSI=%.1f', fontsize=8, inline=True, inline_spacing=4)
cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label('CSI (Critical Success Index)', fontsize=9)

# Frequency Bias isolines
for b, ls, col, lw in [(0.25,'--','#c0392b',1.0), (0.5,'--','#e67e22',1.0),
                        (1.0,'-','#2c3e50',1.8),   (2.0,'--','#e67e22',1.0),
                        (4.0,'--','#c0392b',1.0)]:
    sr_b  = np.linspace(0, 1.0, 200)
    pod_b = b * sr_b
    mask  = pod_b <= 1.0
    ax.plot(sr_b[mask], pod_b[mask], ls=ls, color=col, lw=lw, alpha=0.75)
    valid = np.where(mask)[0]
    if len(valid):
        xi, yi = sr_b[valid[-1]], pod_b[valid[-1]]
        ax.text(xi+0.01, min(yi,0.99), f'B={b}',
                fontsize=7.5, color=col, va='bottom', ha='left')

# ── Threshold sweep using default-weight model ────────────────────────
y_proba_default  = lr_default.predict_proba(X_test)[:, 1]
y_proba_balanced = lr_balanced.predict_proba(X_test)[:, 1]

for proba, color, label in [
    (y_proba_default,  'royalblue',  'LR default — threshold sweep'),
    (y_proba_balanced, 'darkorange', 'LR balanced — threshold sweep'),
]:
    sweep_sr, sweep_pod = [], []
    for t in np.linspace(0.05, 0.95, 80):
        p = (proba >= t).astype(int)
        sweep_sr.append(ps(y_test, p, zero_division=0))
        sweep_pod.append(rs(y_test, p, zero_division=0))
    ax.plot(sweep_sr, sweep_pod, color=color, lw=1.5, alpha=0.5, label=label)

# ── Individual operating points (threshold = 0.5) ─────────────────────
sr_def  = ps(y_test, preds_default,  zero_division=0)
pod_def = rs(y_test, preds_default,  zero_division=0)
sr_bal  = ps(y_test, preds_balanced, zero_division=0)
pod_bal = rs(y_test, preds_balanced, zero_division=0)

ax.scatter(sr_def, pod_def, color='royalblue', s=130, zorder=5,
           label=f'Default  t=0.5  SR={sr_def:.2f} POD={pod_def:.2f}')
ax.scatter(sr_bal, pod_bal, color='darkorange', s=130, marker='D', zorder=5,
           label=f'Balanced t=0.5  SR={sr_bal:.2f} POD={pod_bal:.2f}')
ax.scatter(1.0, 1.0, color='gold', s=200, marker='*', zorder=6,
           edgecolors='black', lw=0.5, label='Perfect model')

ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
ax.set_xlabel('Success Ratio  (= Precision = 1 − FDR)', fontsize=11)
ax.set_ylabel('Probability of Detection  (= Recall / TPR)', fontsize=11)
ax.set_title('Performance Diagram\nGreen = CSI level  |  Lines = Frequency Bias',
             fontsize=12)
ax.legend(loc='lower left', fontsize=8, framealpha=0.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('Reading the diagram:')
print(f'  Default  — SR={sr_def:.3f}, POD={pod_def:.3f}, '
      f'CSI={csi_from_sr_pod(sr_def, pod_def):.3f}, '
      f'Bias={pod_def/sr_def if sr_def>0 else float("nan"):.3f}')
print(f'  Balanced — SR={sr_bal:.3f}, POD={pod_bal:.3f}, '
      f'CSI={csi_from_sr_pod(sr_bal, pod_bal):.3f}, '
      f'Bias={pod_bal/sr_bal if sr_bal>0 else float("nan"):.3f}')
print()
print('The balanced model trades SR (moves left) for higher POD (moves up).')
print('Both threshold sweeps show the full envelope each model can achieve.')
print('Compare the shapes: which model has more room to improve CSI by tuning threshold?')


---

## **1.9 Putting It All Together: A Metric Selection Framework**


| Business Situation | Best Metric(s) |
|--------------------|----------------|
| Balanced classes, no strong error preference | Accuracy, F1-Macro |
| Imbalanced classes | F1 (minority), AUC-ROC, PR-AUC, MCC |
| Missing a positive is very costly (disease, fraud, safety) | **Maximize Recall / POD** |
| A false alarm is very costly (spam, content flags) | **Maximize Precision / SR** |
| Comparing models before choosing a threshold | **AUC-ROC** |
| Highly imbalanced + threshold-free comparison | **PR-AUC** |
| Rare events where non-events dominate (weather, failures) | **CSI, Performance Diagram** |
| Diagnosing over/under-prediction tendency | **Frequency Bias** |
| Single honest number for imbalanced data | **MCC** |
| Comparing multiple models or thresholds visually, rare events | **Performance Diagram** (section 1.8) |

In [ ]:
from sklearn.metrics import average_precision_score

# Final summary: all Week 7 metrics side by side
y_proba_final = lr.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy"  : accuracy_score(y_test, lr_preds),
    "Precision" : precision_score(y_test, lr_preds),
    "Recall"    : recall_score(y_test, lr_preds),
    "F1-Score"  : f1_score(y_test, lr_preds),
    "AUC-ROC"   : roc_auc_score(y_test, y_proba_final),
    "PR-AUC"    : average_precision_score(y_test, y_proba_final),
}

print("=" * 52)
print("  WEEK 7 METRICS SUMMARY — Logistic Regression")
print("=" * 52)
for name, val in metrics.items():
    bar = '█' * int(val * 28)
    print(f"  {name:<12} {val:.4f}  {bar}")
print("=" * 52)
print("\nAccuracy (0.94) flatters the model. Recall and PR-AUC")
print("reveal that performance on the minority class is modest.")

### **1.9.1 Faith Integration**
> *"The one who states his case first seems right, until the other comes and examines him."* — Proverbs 18:17

**Accuracy is the first case.** It sounds compelling. But when we examine it with the full toolkit — precision, recall, CSI, MCC, AUC, performance diagrams — we discover what it hides. As data scientists, we have an **ethical obligation** to choose metrics honestly. A model predicting \"no cancer\" for everyone scores 99% accuracy on a rare-cancer dataset. Reporting that number without context is not just statistically wrong — it can cost lives.

Integrity in metric selection is part of practicing data science faithfully.


---

## **BREAK (10-15 minutes)**

---

## **2.1 Lab Exercises** (new notebook)

---

## **3.1: Review & Wrap-Up**

### **3.1.1 Key Takeaways**

1. **The Finley Fallacy is everywhere** — a model that predicts the majority class constantly can look accurate while being completely useless; accuracy is only trustworthy when classes are balanced

2. **The confusion matrix is your foundation** — every metric is a ratio built from TP, TN, FP, FN; know what question each ratio answers before you compute it

3. **Precision and recall pull in opposite directions** — lowering the threshold raises recall and lowers precision; raising it does the reverse; the tradeoff is unavoidable given a fixed model

4. **Threshold selection is a values decision, not a statistical one** — it reflects the relative cost of false alarms vs. missed detections in your specific domain

5. **F1 is the harmonic mean** — it punishes extreme precision/recall asymmetry; use $F_\beta$ when you need to weight one side more heavily

6. **Specificity is recall for the negative class** — FPR = 1 − Specificity, which is why you've been looking at it all along on the x-axis of ROC curves

7. **FDR = 1 − Precision** — the fraction of your positive predictions that are wrong; central in genomics, clinical trials, and multiple hypothesis testing

8. **CSI excludes true negatives deliberately** — designed for rare events where TN is huge and would otherwise dominate any score; equivalent to Intersection over Union (IoU) of predicted vs. actual positive sets

9. **Bias ≠ Skill** — Frequency Bias = 1 means you predict positive events at the right rate; it says nothing about whether your individual predictions are correct

10. **MCC uses all four confusion matrix cells** — it's a correlation coefficient between predicted and actual labels; the most robust single-number summary for imbalanced data

11. **ROC-AUC originated in WWII signal detection** — it measures the probability that a random positive is scored higher than a random negative; threshold-independent model comparison

12. **ROC-AUC can be misleading on severe imbalance** — the large TN denominator in FPR makes the curve look better than it is; prefer PR-AUC when the positive class is rare

13. **Youden's J = TPR − FPR** — identifies the ROC threshold that maximizes the gap between true and false positive rates; a reasonable default when error costs are unknown

14. **Class weighting shifts the decision boundary** toward the minority class by upweighting its gradient contribution during training; it doesn't change the model's discriminative ability, only its operating point

15. **The Performance Diagram** places Precision (x) vs. Recall (y) with CSI isolines and Frequency Bias isolines overlaid — all four rare-event metrics in one view; ideal point is top-right

16. **Three visualization tools, three perspectives** — ROC curve (includes TN via FPR), PR curve (ignores TN, better for imbalance), performance diagram (rare-event skill with bias context)

17. **Metric choice carries ethical weight** — the wrong metric doesn't just produce bad models; it can produce models that harm people while appearing to perform well



---

## **3.2 Coming Up**

### **3.2.1 Next Week (Week 8): Naïve Bayes & Support Vector Machines**

1. **Naïve Bayes Classifier**
   - Probabilistic classification using Bayes' theorem
   - When and why the 'naïve' independence assumption works in practice
   - Applications in text classification

2. **Support Vector Machines (SVM)**
   - The maximum-margin hyperplane
   - The 'kernel trick' — making non-linear boundaries with linear math
   - When to use SVMs vs. logistic regression

3. **Comparing Classifiers**
   - Using the Week 7 evaluation toolkit to choose between algorithms
   - Structured vs. text data tradeoffs

### **3.2.2 Why This Matters**

The evaluation framework from Week 7 is your permanent toolkit — you'll use precision, recall, AUC, and the confusion matrix every time you build a classifier for the rest of the semester (and your career).

**Prepare by:**
- Completing this week's lab on the Pima Diabetes dataset
- Revisiting the confusion matrix metrics until you can compute them by hand
- Thinking about your final project dataset — what would be the right metric for your problem?

### **3.2.3 Reminders**
- **Lab due:** Monday, 23 March @ 6:00 PM (grace period: Wednesday, 25 March @ 11:59 PM)
- **Spring Break:** 16–20 March — no class!
- **Project Check-in Due:** Monday, 23 March (first class after Spring Break)
